<a href="https://colab.research.google.com/github/hemasri159/Practical-Experiments_ML/blob/main/WEEK_8_practical.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q xgboost

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import time
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

from xgboost import XGBClassifier

In [ ]:
DATA_PATH = "/content/drive/MyDrive/colab/placement_predict_50k_adjusted.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully!")
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

Dataset loaded successfully!
Shape: (50000, 21)

Columns:
['Gender', 'City', 'CollegeTier', 'Stream', 'Specialisation', 'Hostel', 'HistoryOfBacklogs', 'CGPA', 'AttendancePercent', 'Internships', 'Projects', 'Workshops', 'Certifications', 'Publications', 'AptitudeTestScore', 'SoftSkillsRating', 'CodingTestScore', 'MockInterviewScore', 'ExtraCurricular', 'PlacementStatus', 'IsAnomaly']

First 5 rows:


,Gender,City,CollegeTier,Stream,Specialisation,Hostel,HistoryOfBacklogs,CGPA,AttendancePercent,Internships,...,Workshops,Certifications,Publications,AptitudeTestScore,SoftSkillsRating,CodingTestScore,MockInterviewScore,ExtraCurricular,PlacementStatus,IsAnomaly
0,Female,Delhi,Tier3,IT,DataScience,Yes,Yes,6.63,68.3,2,...,0.0,1,0,62.3,6.57,40.6,65.7,No,0,0
1,Male,Chennai,Tier2,ECE,AI,Yes,No,6.40,71.0,1,...,0.0,2,0,44.0,5.86,40.3,51.8,No,0,0
2,Female,Hyderabad,Tier3,ECE,Networking,No,No,7.73,75.1,1,...,2.0,2,1,73.8,7.50,73.6,67.9,No,1,0
3,Female,Jaipur,Tier3,ECE,Embedded,No,No,9.73,99.2,4,...,5.0,6,2,100.0,9.41,98.7,NaN,No,1,0
4,Male,Ahmedabad,Tier3,Mechanical,DataScience,No,No,9.01,99.6,2,...,NaN,4,2,90.8,9.24,83.1,100.0,No,1,0


In [ ]:
RANDOM_STATE = 42
TARGET_COL = "PlacementStatus"

if "IsAnomaly" in df.columns:
    df = df.drop(columns=["IsAnomaly"])

y = df[TARGET_COL].astype(int)

X = df.drop(columns=[TARGET_COL])

cat_cols = X.select_dtypes(
    exclude="number"
).columns.tolist()

num_cols = X.select_dtypes(
    include="number"
).columns.tolist()

print("Categorical columns:")
print(cat_cols)

print("\nNumerical columns:")
print(num_cols)

Categorical columns:
['Gender', 'City', 'CollegeTier', 'Stream', 'Specialisation', 'Hostel', 'HistoryOfBacklogs', 'ExtraCurricular']

Numerical columns:
['CGPA', 'AttendancePercent', 'Internships', 'Projects', 'Workshops', 'Certifications', 'Publications', 'AptitudeTestScore', 'SoftSkillsRating', 'CodingTestScore', 'MockInterviewScore']


In [ ]:
encoders = {}

for c in cat_cols:
    le = LabelEncoder()
    X[c] = le.fit_transform(X[c].astype(str))
    encoders[c] = le

print("Categorical columns encoded successfully.")

Categorical columns encoded successfully.


In [ ]:
imputer = SimpleImputer(strategy="median")

X[num_cols] = imputer.fit_transform(X[num_cols])

print("Missing values handled successfully.")

Missing values handled successfully.


In [ ]:
scaler = StandardScaler()

X[num_cols] = scaler.fit_transform(X[num_cols])

print("Numerical columns scaled successfully.")

Numerical columns scaled successfully.


In [ ]:
VAL_SIZE = 0.15
TEST_SIZE = 0.15

X_train_val, X_test, y_train_val, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE
)

val_ratio = VAL_SIZE / (1 - TEST_SIZE)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val,
    y_train_val,
    test_size=val_ratio,
    stratify=y_train_val,
    random_state=RANDOM_STATE
)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

Train: (34999, 19)
Validation: (7501, 19)
Test: (7500, 19)


In [ ]:
ada_base = DecisionTreeClassifier(
    max_depth=2,
    random_state=RANDOM_STATE
)

ada = AdaBoostClassifier(
    estimator=ada_base,
    n_estimators=200,
    learning_rate=0.5,
    random_state=RANDOM_STATE
)

print("AdaBoost model created.")

AdaBoost model created.


In [ ]:
t0 = time.time()

ada.fit(
    X_train,
    y_train
)

ada_fit_time = time.time() - t0

print("AdaBoost training completed.")
print("Training time:", round(ada_fit_time, 2), "seconds")

AdaBoost training completed.
Training time: 11.5 seconds


In [ ]:
ada_val_pred = ada.predict(X_val)

ada_val_proba = ada.predict_proba(X_val)[:, 1]

print("AdaBoost prediction completed.")

AdaBoost prediction completed.


In [ ]:
ada_accuracy = accuracy_score(
    y_val,
    ada_val_pred
)

ada_f1 = f1_score(
    y_val,
    ada_val_pred
)

ada_roc_auc = roc_auc_score(
    y_val,
    ada_val_proba
)

print("========== ADABOOST RESULTS ==========")
print("Accuracy:", round(ada_accuracy, 4))
print("F1 Score:", round(ada_f1, 4))
print("ROC-AUC:", round(ada_roc_auc, 4))
print("Trees:", ada.n_estimators)
print("Training Time:", round(ada_fit_time, 2), "seconds")

========== ADABOOST RESULTS ==========
Accuracy: 0.7963
F1 Score: 0.781
ROC-AUC: 0.8802
Trees: 200
Training Time: 11.5 seconds


In [ ]:
xgb = XGBClassifier(
    n_estimators=1000,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    early_stopping_rounds=30,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

print("XGBoost model created.")

XGBoost model created.


In [ ]:
t0 = time.time()

xgb.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
)

xgb_fit_time = time.time() - t0

print("XGBoost training completed.")
print("Training time:", round(xgb_fit_time, 2), "seconds")

XGBoost training completed.
Training time: 2.39 seconds


In [ ]:
xgb_val_pred = xgb.predict(X_val)

xgb_val_proba = xgb.predict_proba(X_val)[:, 1]

print("XGBoost prediction completed.")

XGBoost prediction completed.


In [ ]:
xgb_accuracy = accuracy_score(
    y_val,
    xgb_val_pred
)

xgb_f1 = f1_score(
    y_val,
    xgb_val_pred
)

xgb_roc_auc = roc_auc_score(
    y_val,
    xgb_val_proba
)

print("========== XGBOOST RESULTS ==========")
print("Accuracy:", round(xgb_accuracy, 4))
print("F1 Score:", round(xgb_f1, 4))
print("ROC-AUC:", round(xgb_roc_auc, 4))
print("Trees:", xgb.best_iteration + 1)
print("Training Time:", round(xgb_fit_time, 2), "seconds")

========== XGBOOST RESULTS ==========
Accuracy: 0.7959
F1 Score: 0.7827
ROC-AUC: 0.8826
Trees: 135
Training Time: 2.39 seconds


In [ ]:
results = [
    {
        "Model": "AdaBoost",
        "Accuracy": ada_accuracy,
        "F1 Score": ada_f1,
        "ROC-AUC": ada_roc_auc,
        "Trees": ada.n_estimators,
        "Time (sec)": round(ada_fit_time, 2)
    },
    {
        "Model": "XGBoost",
        "Accuracy": xgb_accuracy,
        "F1 Score": xgb_f1,
        "ROC-AUC": xgb_roc_auc,
        "Trees": xgb.best_iteration + 1,
        "Time (sec)": round(xgb_fit_time, 2)
    }
]

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    "Accuracy",
    ascending=False
).reset_index(drop=True)

display(results_df)

,Model,Accuracy,F1 Score,ROC-AUC,Trees,Time (sec)
0,AdaBoost,0.796294,0.780963,0.880162,200,11.50
1,XGBoost,0.795894,0.782744,0.882638,135,2.39


In [ ]:
best_model = results_df.iloc[0]["Model"]

print("======================================")
print("FINAL CONCLUSION")
print("======================================")
print("Best model based on validation accuracy:", best_model)
print()
print(results_df.to_string(index=False))

FINAL CONCLUSION
Best model based on validation accuracy: AdaBoost

   Model  Accuracy  F1 Score  ROC-AUC  Trees  Time (sec)
AdaBoost  0.796294  0.780963 0.880162    200       11.50
 XGBoost  0.795894  0.782744 0.882638    135        2.39
